<a href="https://colab.research.google.com/github/pbae123/DataClub-Meta/blob/main/MetaSetFit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Setup

In [1]:
#install setfit library + sentence-transformers, datasets, and scikit-learn
#go to runtime --> change runtime type --> switch to T4GPU
!pip install setfit==1.1.0 transformers==4.44.2 sentence-transformers==3.0.1 datasets==2.21.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 142.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


#2. upload dataset

In [2]:
from google.colab import files
#upload directly from computer (navigate to aggregatedSet.txt)
uploaded = files.upload()

Saving aggregatedSet.txt to aggregatedSet.txt


#3. fix syntax

In [3]:
'''
- since our dataset is JSONL (one JSON object per line) with this struture {"text"... "labels"...}
- we need to convert each label dict into a the format setfit expects... [0,1,1,0,0]
'''

import json
import numpy as np
from datasets import Dataset

# ── Load the data ──
DATA_PATH = "aggregatedSet.txt"
records = []
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Fix curly/smart quotes that Excel sometimes introduces
        line = line.replace("\u201c", '"').replace("\u201d", '"')
        line = line.replace("\u2018", "'").replace("\u2019", "'")
        records.append(json.loads(line))

print(f"Loaded {len(records)} labeled posts")

#label columns
features = ["Self-Harm", "Depression", "Aggression", "Anxiety", "Sexually suggestive"]

# ── Build HuggingFace Dataset ──
texts = [r["text"] for r in records]
labels = [[r["labels"][f] for f in features] for r in records]

dataset = Dataset.from_dict({"text": texts, "labels": labels})
print(dataset)
print(f"\nLabel order: {features}")

Loaded 451 labeled posts
Dataset({
    features: ['text', 'labels'],
    num_rows: 451
})

Label order: ['Self-Harm', 'Depression', 'Aggression', 'Anxiety', 'Sexually suggestive']


#4. label distribution

In [4]:
#see how many posts have each label + multilabeled
import pandas as pd

# Count per label
label_counts = {}
for f_idx, f_name in enumerate(features):
    label_counts[f_name] = sum(1 for lab in labels if lab[f_idx] == 1)

# Count multilabel posts
single_count = sum(1 for lab in labels if sum(lab) == 1)
multi_count = sum(1 for lab in labels if sum(lab) > 1)
zero_count = sum(1 for lab in labels if sum(lab) == 0)

print("=== Label Distribution ===")
for name, count in label_counts.items():
    print(f"  {name}: {count} ({count/len(labels)*100:.1f}%)")

print(f"\n  Posts with one label: {single_count}")
print(f"\n  Posts with multiple labels: {multi_count}")
print(f"  Posts with NO labels (all zeros): {zero_count}")
print(f"  Total posts: {len(labels)}")

=== Label Distribution ===
  Self-Harm: 58 (12.9%)
  Depression: 105 (23.3%)
  Aggression: 47 (10.4%)
  Anxiety: 76 (16.9%)
  Sexually suggestive: 83 (18.4%)

  Posts with one label: 230

  Posts with multiple labels: 64
  Posts with NO labels (all zeros): 157
  Total posts: 451


#5. train + test slit

In [5]:
#will use a 80/20 split. so split for evaluation and testing
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Training set: {len(train_dataset)} posts")
print(f"Evaluation set:  {len(eval_dataset)} posts")

Training set: 360 posts
Evaluation set:  91 posts


#6. load setfit model

In [6]:
#loading a pretrained sentence transformer and configure for multi-label classification
from setfit import SetFitModel

model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2",
    multi_target_strategy="one-vs-rest",
)
#note. a warning about "initialising classification head with random weights" is expected. that's why we are training it

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


#7. Train model

In [8]:
'''
what set fit does:
1. Contrastive learning — fine-tunes the Sentence Transformer body by generating text pairs (similar/dissimilar based on shared labels)
2. Classification head — trains the `one-vs-rest` logistic regression head on the resulting embeddings
'''
from setfit import Trainer, TrainingArguments
from sentence_transformers.losses import CosineSimilarityLoss

args = TrainingArguments(
    batch_size=32,
    num_epochs=1,
    num_iterations=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    column_mapping={"text": "text", "labels": "label"},
)

# train
trainer.train()

#training loss: 0.027000, validation loss: 0.113805

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
/usr/local/lib/python3.12/dist-packages/datasets/utils/_dill.py:385: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  obj.co_lnotab,  # for < python 3.10 [not counted in args]
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Map:   0%|          | 0/360 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 14400
  Batch size = 32
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Epoch,Training Loss,Validation Loss
1,0.027000,0.113805


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/jupyter_client/sessi

#8. evaluate

In [10]:
metrics = trainer.evaluate(eval_dataset)
print("=== Evaluation Results ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")


from sklearn.metrics import classification_report

eval_texts = eval_dataset["text"]
eval_labels = eval_dataset["labels"]
preds = model.predict(eval_texts)

y_true = np.array(eval_labels)
y_pred = np.array(preds.tolist() if hasattr(preds, 'tolist') else preds)

print(classification_report(y_true, y_pred, target_names=features, zero_division=0))

Applying column mapping to the evaluation dataset
***** Running evaluation *****


=== Evaluation Results ===
  accuracy: 0.7143


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                     precision    recall  f1-score   support

          Self-Harm       0.88      0.88      0.88         8
         Depression       0.59      1.00      0.74        17
         Aggression       0.83      1.00      0.91         5
            Anxiety       0.59      0.83      0.69        12
Sexually suggestive       0.93      0.78      0.85        18

          micro avg       0.71      0.88      0.79        60
          macro avg       0.76      0.90      0.81        60
       weighted avg       0.75      0.88      0.79        60
        samples avg       0.49      0.51      0.49        60



/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


#9.save trained model

In [14]:
# Save locally (will create a folder with model files)
model.save_pretrained("setfit-whisper-multilabel")

print("Model saved to ./setfit-whisper-multilabel/")
#To download: use the file browser on the left → right-click the folder → Download

Model saved to ./setfit-whisper-multilabel/
